# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [ ]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [ ]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [ ]:
!java DemurrageLegacy.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [1]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

NameError: name 'pd' is not defined

### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** `static` здесь не является дефектом: метод не использует состояние экземпляра, поэтому `static` корректен и даже предпочтителен — он явно показывает, что метод не зависит от объекта. Замечание было бы вкусовым (`nit`), а эталон демонстрирует, что вкусовые замечания без обоснования в разбор не включаются. Если бы в проекте был принят стиль «все методы — экземплярные» (например, для тестируемости через подмену), замечание можно было бы сформулировать как `nit` с обоснованием, но это уже вопрос соглашений команды, а не корректности.

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [ ]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

In [11]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Error: Could not find or load main class ShiftPlanLegacy.java
Caused by: java.lang.ClassNotFoundException: ShiftPlanLegacy.java


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [10]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(Строка="24",
         Категория="Корректность",
         Серьёзность="blocker",
         Замечание="Удаление элемента из ArrayList внутри цикла for-each приводит к ConcurrentModificationException: при данных из main программа падает при первом же удалении (Иванов, 11.0 ч). Кроме того, даже без исключения удаление сдвигает индексы и часть элементов будет пропущена.",
         Предложение="Собрать отфильтрованный список в новый ArrayList (brigades.stream().filter(...).collect(...)) либо использовать Iterator.remove()."),
    dict(Строка="35-37",
         Категория="Поведение при ошибках",
         Серьёзность="blocker",
         Замечание="Если после фильтрации бригад меньше, чем need, вызов brigades.get(i) бросает IndexOutOfBoundsException вместо осмысленного сообщения. Сейчас при need=3 и трёх отфильтрованных бригадах программа упадёт.",
         Предложение="Проверять need <= brigades.size() и при нехватке бросать IllegalArgumentException с указанием, сколько бригад доступно и сколько запрошено."),
    dict(Строка="40-44",
         Категория="Поведение при ошибках",
         Серьёзность="blocker",
         Замечание="FileWriter не закрывается: catch (Exception e) {} проглатывает любую ошибку ввода-вывода молча. Файл остаётся открытым, при ошибке записи никто об этом не узнает, а наряд может оказаться неполным.",
         Предложение="Использовать try-with-resources для FileWriter и либо пробрасывать IOException, либо логировать её и сообщать вызывающему коду."),
    dict(Строка="40-44",
         Категория="Поведение при ошибках",
         Серьёзность="major",
         Замечание="Пустой catch (Exception e) {} — классическое «проглоченное исключение». Любая проблема с файлом (нет прав, диск заполнен) останется незамеченной, а метод вернёт результат, будто запись удалась.",
         Предложение="Заменить пустой catch на осмысленную обработку: обернуть в RuntimeException с сообщением или пробросить IOException в сигнатуре метода."),
    dict(Строка="24",
         Категория="Корректность",
         Серьёзность="major",
         Замечание="Сравнение вещественного hoursPerShift > 10.0 с константой без допуска: при значении, полученном из расчёта (например, 10.0000000001), бригада будет ошибочно отфильтрована. Порог 10.0 — магическое число, не привязанное к документу.",
         Предложение="Вынести порог в именованную константу MAX_HOURS_PER_SHIFT и при необходимости использовать допуск (Math.abs(hours - MAX) > EPS) либо BigDecimal для часов."),
    dict(Строка="9-10, 43",
         Категория="Данные и безопасность",
         Серьёзность="major",
         Замечание="Статические поля SimpleDateFormat и log не потокобезопасны: SimpleDateFormat при одновременном вызове из двух потоков портит результат, а ArrayList.log теряет записи при конкурентном add. Это скрытый дефект, который проявится только под нагрузкой.",
         Предложение="Сделать log локальной переменной или использовать Collections.synchronizedList, а SimpleDateFormat заменить на java.time.format.DateTimeFormatter (потокобезопасен)."),
    dict(Строка="43",
         Категория="Данные и безопасность",
         Серьёзность="major",
         Замечание="В журнал пишется только факт «наряд сформирован», но не состав наряда и не отфильтрованные бригады. При разборе инцидента нельзя понять, кто попал в наряд, а кто нет — журнал не выполняет свою функцию.",
         Предложение="Логировать состав наряда и причины отсева каждой бригады (часы > нормы, не допущена) с указанием фамилии."),
    dict(Строка="35-37",
         Категория="Проектирование",
         Серьёзность="major",
         Замечание="Конкатенация строк в цикле result = result + ... создаёт новый объект String на каждой итерации; при большом need это квадратичная сложность и лишняя нагрузка на GC.",
         Предложение="Использовать StringBuilder и добавлять фамилии через append."),
    dict(Строка="24",
         Категория="Читаемость",
         Серьёзность="minor",
         Замечание="Сравнение brigade.admitted == false избыточно и хуже читается, чем !brigade.admitted; смешение стилей в одном условии (hoursPerShift > 10.0 || admitted == false) затрудняет восприятие.",
         Предложение="Заменить на !brigade.admitted и, при желании, вынести условие в метод isEligible(Brigade)."),
    dict(Строка="29-37",
         Категория="Тесты",
         Серьёзность="major",
         Замечание="Для метода make нет ни одного теста, хотя он содержит фильтрацию, запись в файл, побочный эффект в статическом журнале и потенциальные исключения. Без характеризующего теста нельзя доказать, что рефакторинг не изменил поведение.",
         Предложение="Добавить характеризующий тест: зафиксировать ожидаемый результат «Иванов; Козлов; » для исходных данных и проверить его после рефакторинга."),
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
print("По серьёзности:", ревью["Серьёзность"].value_counts().to_dict())
ревью

Записано замечаний: 10
По серьёзности: {'major': 6, 'blocker': 3, 'minor': 1}


,Строка,Категория,Серьёзность,Замечание,Предложение
0,24,Корректность,blocker,Удаление элемента из ArrayList внутри цикла fo...,Собрать отфильтрованный список в новый ArrayLi...
1,35-37,Поведение при ошибках,blocker,"Если после фильтрации бригад меньше, чем need,...",Проверять need <= brigades.size() и при нехват...
2,40-44,Поведение при ошибках,blocker,FileWriter не закрывается: catch (Exception e)...,Использовать try-with-resources для FileWriter...
3,40-44,Поведение при ошибках,major,Пустой catch (Exception e) {} — классическое «...,Заменить пустой catch на осмысленную обработку...
4,24,Корректность,major,Сравнение вещественного hoursPerShift > 10.0 с...,Вынести порог в именованную константу MAX_HOUR...
5,"9-10, 43",Данные и безопасность,major,Статические поля SimpleDateFormat и log не пот...,Сделать log локальной переменной или использов...
6,43,Данные и безопасность,major,В журнал пишется только факт «наряд сформирова...,Логировать состав наряда и причины отсева кажд...
7,35-37,Проектирование,major,Конкатенация строк в цикле result = result + ....,Использовать StringBuilder и добавлять фамилии...
8,24,Читаемость,minor,Сравнение brigade.admitted == false избыточно ...,"Заменить на !brigade.admitted и, при желании, ..."
9,29-37,Тесты,major,"Для метода make нет ни одного теста, хотя он с...",Добавить характеризующий тест: зафиксировать о...


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [12]:
%%writefile ShiftPlanFixed.java
import java.io.IOException;
import java.io.Writer;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.Paths;
import java.time.LocalDate;
import java.time.format.DateTimeFormatter;
import java.util.ArrayList;
import java.util.List;

public class ShiftPlanFixed {

    // Замечание 5: порог вынесен в именованную константу.
    static final double MAX_HOURS_PER_SHIFT = 10.0;
    // Замечание 6: DateTimeFormatter потокобезопасен (замена SimpleDateFormat).
    static final DateTimeFormatter DATE_FORMAT = DateTimeFormatter.ofPattern("dd.MM.yyyy");

    private static int failedChecks = 0;

    public static void main(String[] args) {
        // Замечание 1: ConcurrentModificationException устранён — фильтрация в новый список.
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        List<String> journal = new ArrayList<>();
        String result = make(brigades, 2, Paths.get("наряд.txt"), journal);
        System.out.println("Наряд: " + result);
        System.out.println("Журнал: " + journal);

        // Замечание 10: характеризующий тест на исходных данных.
        // Иванов отсеян (11 ч > 10), Петров оставлен (0 ч, допущен),
        // Сидоров отсеян (не допущен), Козлов оставлен (8 ч, допущен).
        // При need=2 наряд = "Петров; Козлов; ".
        check("состав наряда при need=2: 'Петров; Козлов; '",
              "Петров; Козлов; ".equals(result));
        check("Иванов отсеян (11 ч > 10)", !result.contains("Иванов"));
        check("Сидоров отсеян (не допущен)", !result.contains("Сидоров"));

        // Замечание 2: need > size — осмысленное исключение, а не IndexOutOfBounds.
        List<Brigade> copy = new ArrayList<>();
        copy.add(new Brigade("Петров", 0.0, true));
        boolean threw = false;
        try {
            make(copy, 3, Paths.get("наряд2.txt"), new ArrayList<>());
        } catch (IllegalArgumentException e) {
            threw = true;
            System.out.println("Осмысленное исключение: " + e.getMessage());
        }
        check("need > size бросает IllegalArgumentException", threw);

        // Замечание 6/7: журнал содержит состав и причины отсева.
        List<Brigade> copy2 = new ArrayList<>();
        copy2.add(new Brigade("Иванов", 11.0, true));
        copy2.add(new Brigade("Петров", 0.0, true));
        List<String> journal2 = new ArrayList<>();
        make(copy2, 1, Paths.get("наряд3.txt"), journal2);
        String allLog = String.join(" | ", journal2);
        check("журнал содержит причину отсева Иванова",
              allLog.contains("Иванов") && allLog.contains("10"));
        check("журнал содержит дату в формате dd.MM.yyyy",
              allLog.matches(".*\\d{2}\\.\\d{2}\\.\\d{4}.*"));

        // Замечание 9: !admitted вместо == false.
        List<Brigade> copy3 = new ArrayList<>();
        copy3.add(new Brigade("Сидоров", 4.5, false));
        copy3.add(new Brigade("Козлов", 8.0, true));
        List<String> journal3 = new ArrayList<>();
        String result3 = make(copy3, 1, Paths.get("наряд4.txt"), journal3);
        check("недопущенный Сидоров не попал в наряд",
              "Козлов; ".equals(result3));

        // Замечание 4: пустой catch устранён — ошибка записи пробрасывается.
        boolean ioThrew = false;
        try {
            make(copy3, 1, Paths.get("/nonexistent_dir/наряд.txt"), new ArrayList<>());
        } catch (RuntimeException e) {
            ioThrew = true;
            System.out.println("Ошибка записи проброшена: " + e.getClass().getSimpleName());
        }
        check("ошибка записи файла не проглатывается", ioThrew);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // Замечание 3: try-with-resources + проброс IOException в RuntimeException.
    // Замечание 8: StringBuilder вместо конкатенации в цикле.
    // Замечание 1: фильтрация в новый список, исходный не мутируется.
    public static String make(List<Brigade> brigades, int need, Path outputPath, List<String> journal) {
        if (brigades == null) {
            throw new IllegalArgumentException("список бригад не задан");
        }
        if (need < 0) {
            throw new IllegalArgumentException("need не может быть отрицательным: " + need);
        }
        if (journal == null) {
            throw new IllegalArgumentException("журнал не задан");
        }

        List<Brigade> eligible = new ArrayList<>();
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > MAX_HOURS_PER_SHIFT) {
                journal.add(DATE_FORMAT.format(LocalDate.now())
                        + " отсев: " + brigade.lastName
                        + " (" + brigade.hoursPerShift + " ч > " + MAX_HOURS_PER_SHIFT + ")");
            } else if (!brigade.admitted) {
                journal.add(DATE_FORMAT.format(LocalDate.now())
                        + " отсев: " + brigade.lastName + " (не допущен)");
            } else {
                eligible.add(brigade);
            }
        }

        if (need > eligible.size()) {
            throw new IllegalArgumentException(
                    "запрошено бригад: " + need + ", доступно после отсева: " + eligible.size());
        }

        StringBuilder resultBuilder = new StringBuilder();
        for (int i = 0; i < need; i++) {
            resultBuilder.append(eligible.get(i).lastName).append("; ");
        }
        String result = resultBuilder.toString();

        try (Writer writer = Files.newBufferedWriter(outputPath, StandardCharsets.UTF_8)) {
            writer.write(result);
        } catch (IOException e) {
            throw new RuntimeException("не удалось записать наряд в " + outputPath, e);
        }

        journal.add(DATE_FORMAT.format(LocalDate.now()) + " наряд сформирован: " + result);
        return result;
    }

    static void check(String name, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + name);
        if (!passed) {
            failedChecks++;
        }
    }
}

class Brigade {
    final String lastName;
    final double hoursPerShift;
    final boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Overwriting ShiftPlanFixed.java


In [13]:
!rm -f ShiftPlanLegacy.class Brigade.class ShiftPlanLegacy.java
!java ShiftPlanFixed.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Наряд: Петров; Козлов; 
Журнал: [23.11.2026 отсев: Иванов (11.0 ч > 10.0), 23.11.2026 отсев: Сидоров (не допущен), 23.11.2026 наряд сформирован: Петров; Козлов; ]
[OK]   состав наряда при need=2: 'Петров; Козлов; '
[OK]   Иванов отсеян (11 ч > 10)
[OK]   Сидоров отсеян (не допущен)
Осмысленное исключение: запрошено бригад: 3, доступно после отсева: 1
[OK]   need > size бросает IllegalArgumentException
[OK]   журнал содержит причину отсева Иванова
[OK]   журнал содержит дату в формате dd.MM.yyyy
[OK]   недопущенный Сидоров не попал в наряд
Ошибка записи проброшена: RuntimeException
[OK]   ошибка записи файла не проглатывается

Все проверки пройдены.


**Таблица прослеживаемости.** Каждое изменение кода прослеживается до строки таблицы замечаний.

| № | Строка | Серьёзность | Что изменено в коде |
|---|---|---|---|
| 1 | 24 | blocker | Фильтрация вынесена в новый список `eligible`, исходный `brigades` не мутируется; `ConcurrentModificationException` устранён. |
| 2 | 35-37 | blocker | Добавлена проверка `need > eligible.size()` → `IllegalArgumentException` с указанием доступного количества. |
| 3 | 40-44 | blocker | `try-with-resources` для `Writer`, `IOException` пробрасывается в `RuntimeException`. |
| 4 | 40-44 | major | Пустой `catch` удалён; ошибка записи больше не проглатывается. |
| 5 | 24 | major | Порог вынесен в `MAX_HOURS_PER_SHIFT = 10.0`. |
| 6 | 9-10, 43 | major | `SimpleDateFormat` → `DateTimeFormatter`; `log` стал параметром `journal`, статическое состояние убрано. |
| 7 | 43 | major | Журнал теперь содержит состав наряда и причины отсева каждой бригады. |
| 8 | 35-37 | major | Конкатенация строк заменена на `StringBuilder`. |
| 9 | 24 | minor | `admitted == false` → `!brigade.admitted`. |
| 10 | 29-37 | major | Добавлены характеризующие тесты в `main` (8 проверок `[OK]`/`[FAIL]`). |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [14]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = "https://github.com/den-matyushkov/Matyushkov-GI-41/pull/7"      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = "https://github.com/ilya-charachkin/CharachkinGI-41/pull/7"         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 5       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 3        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 2         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: https://github.com/den-matyushkov/Matyushkov-GI-41/pull/7
Свой запрос: https://github.com/ilya-charachkin/CharachkinGI-41/pull/7


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? |  |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? |  |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? |  |
| Какое из полученных вами замечаний вы отклонили и на каком основании? |  |

### 7.2 Контрольные вопросы

1. **Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.**
   **Ответ:** Ревью — чтение кода человеком до слияния; оно ловит неверно понятое требование, неудачное проектирование, неинформативные имена. Тестирование проверяет заявленное поведение на наборе входов и не видит дефектов проектирования. Статический анализ ищет шаблонные ошибки (неиспользуемые переменные, `==` для строк, потенциальные NPE) без запуска. Ни одно не заменяет остальные: тест не поймёт, что требование неверно; анализатор не оценит архитектуру; ревью не прогонит тысячи входов. Дешевле всего статический анализ (автоматический), затем ревью (минуты человека), затем тестирование (часы прогонов), дороже всего — исправление после выпуска.

2. **Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?**
   **Ответ:** Лучше человека: шаблонные ошибки — `==` для строк, пустой `catch`, незакрытые ресурсы, потенциальный NPE, неиспользуемые импорты, сравнение вещественных на равенство, SQL-конкатенацию. Не находит никогда: неверно понятое бизнес-требование, неудачное имя, нарушение единственной ответственности, выбор алгоритма, архитектурные решения.

3. **Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.**
   **Ответ:** Три части: место (файл/строка), последствие/нарушенный принцип, предложение. Пример переформулировки: «Это ужасно» → «Строка 24: удаление из `ArrayList` внутри `for-each` бросает `ConcurrentModificationException` — при данных из `main` программа падает. Предлагаю фильтровать в новый список».

4. **Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?**
   **Ответ:** Отсутствие ревью оставляет дефект, но не создаёт ложного ощущения проверенности. `Approve` — это явная подпись «проверено», после которой дефект уходит в основную ветку с социальной легитимацией; найти его потом сложнее, а ответственность размыта.

5. **Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?**
   **Ответ:** Характеризующий тест фиксирует текущее поведение кода (даже неверное) до рефакторинга, чтобы после правок доказать, что поведение при корректных данных не изменилось. Без него рефакторинг чужого кода — это правка вслепую.

6. **Почему крупный pull request получает менее качественное ревью, чем несколько мелких?**
   **Ответ:** Внимание рецензента ограничено; на 1000 строк его не хватает на каждую, и он просматривает поверхностно. Несколько мелких PR позволяют глубоко разобрать каждый и быстрее получить обратную связь.

### 7.3 Три вывода по работе

1. Плотность замечаний важнее их количества: 9–10 точных замечаний на 30–40 строк работают, 30 — обесценивают ревью.
2. Самые опасные дефекты — те, что не видны при запуске: потокобезопасность, проглоченные исключения, отсутствие характеризующих тестов. Их находит только чтение.
3. Формулировка «место + последствие + предложение» превращает замечание из претензии в задачу, которую автор может выполнить.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [15]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = "Чарачкин Ильа Александрович"  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Code review»

| Параметр | Значение |
|---|---|
| Исполнитель | Чарачкин Ильа Александрович, группа ГИ-41 |
| Дата отчёта (UTC) | 2026-11-23 09:32 |
| Таблица замечаний | записей 10, заполненных 10, весомых 9 |
| `ShiftPlanFixed.java` | пройдено — самопроверок пройдено: 8 |
| Ревью на чужой запрос | https://github.com/den-matyushkov/Matyushkov-GI-41/pull/7 |
| Свой запрос | https://github.com/ilya-charachkin/CharachkinGI-41/pull/7 |
| Замечаний оставлено / получено / принято | 5 / 3 / 2 |
| Таблица прослеживаемости | заполнена |
| Вопросы и выводы | заполнены |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.
